<a href="https://colab.research.google.com/github/montserrat96zaragoza-glitch/Proyecto-Google-Cloud-Negocio-de-Tamales/blob/main/ETL_Negocio%20Tamales_de_Estandarizaci%C3%B3n_y_Limpieza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Defición de Librerias






In [ ]:

import os
import pandas as pd
import sqlite3
import re
import openpyxl
from google.colab import auth
auth.authenticate_user()
from google.colab import drive
from google.cloud import storage
drive.mount('/content/drive')
import glob

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configuración de archivos y bucket

In [ ]:

CARPETA_LOCAL = "/content/drive/MyDrive/Proyecto Google/MaterialesProyectoFinal/MaterialesProyectoFinal"
CARPETA_CSV = os.path.join(CARPETA_LOCAL, "csv_limpios")
os.makedirs(CARPETA_CSV, exist_ok=True)

project_ID="tamalesdonrutilio"
BUCKET_NAME = "bucket_rutilio"
Destino = "datos/Base_general.csv"

In [ ]:


client = storage.Client(project=project_ID)

# Obtener lista de buckets
buckets = [b.name for b in client.list_buckets()]

if BUCKET_NAME in buckets:
    print(f"✅ El bucket '{BUCKET_NAME}' ya existe.")
else:
    bucket = client.bucket(BUCKET_NAME)
    bucket.storage_class = "STANDARD"

    bucket.create(location="us-central1")

    print(f"🆕 Bucket '{BUCKET_NAME}' creado correctamente.")


✅ El bucket 'bucket_rutilio' ya existe.


## Homologación de formato a csv y nombre de las bases

In [ ]:


# Contador
contador_base = 1  # Para Base1, Base2, Base3...

# -----------------------------
# PROCESAMIENTO
# -----------------------------
for archivo in os.listdir(CARPETA_LOCAL):
    ruta = os.path.join(CARPETA_LOCAL, archivo)

    if os.path.isdir(ruta):
        continue

    try:
        archivo_lower = archivo.lower()

        # ---------------- CSV ----------------
        if archivo_lower.endswith(".csv"):
            with open(ruta, 'r', encoding='utf-8', errors='replace') as f:
                lineas = [l.strip() for l in f if l.strip()]

            def split_line(line):
                line = line.strip('"')
                return re.split(r'\t|;\s*|,\s*|\|\s*', line)

            data = [split_line(l) for l in lineas]

            max_cols = max(len(row) for row in data)
            for row in data:
                row.extend([None] * (max_cols - len(row)))

            encabezado = data[0]
            df = pd.DataFrame(data[1:], columns=encabezado, dtype=str)

        # ---------------- TXT ----------------
        elif archivo_lower.endswith(".txt"):
            with open(ruta, 'r', encoding='utf-8', errors='replace') as f:
                lineas = [l.strip() for l in f if l.strip()]

            def split_line(line):
                line = line.strip('"')
                return re.split(r'\t|;\s*|\|\s*', line)

            data = [split_line(l) for l in lineas]

            max_cols = max(len(row) for row in data)
            for row in data:
                row.extend([None] * (max_cols - len(row)))

            encabezado = data[0]
            df = pd.DataFrame(data[1:], columns=encabezado, dtype=str)

        # ---------------- XLSX ----------------
        elif archivo_lower.endswith(".xlsx"):
            hojas = pd.read_excel(ruta, sheet_name=None, dtype=str)

            for _, df in hojas.items():
                break  # solo la primera hoja

        # ---------------- SQLITE / DB ----------------
        elif archivo_lower.endswith(".sqlite") or archivo_lower.endswith(".db"):
            conn = sqlite3.connect(ruta)
            tablas = pd.read_sql(
                "SELECT name FROM sqlite_master WHERE type='table';",
                conn
            )

            tabla = tablas['name'][0]
            df = pd.read_sql(f"SELECT * FROM {tabla}", conn, dtype=str)
            conn.close()

        else:
            print(f"Tipo no soportado: {archivo}")
            continue

        # ---------------- LIMPIEZA COMÚN ----------------
        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
            .str.replace(r"[^a-z0-9_]", "", regex=True)
        )

        for col in df.columns:
            df[col] = df[col].astype(str).str.strip()

        # ---------GUARDAR CSV y homologar nombre de las base ----------------
        nombre_csv = f"Base{contador_base}.csv"
        ruta_destino = os.path.join(CARPETA_CSV, nombre_csv)

        df.to_csv(ruta_destino, index=False, encoding="utf-8")
        print(f"Creado: {nombre_csv}")

        contador_base += 1

    except Exception as e:
        print(f"Error con {archivo}: {e}")

print("Todos los archivos convertidos a CSV limpio")


Creado: Base1.csv
Creado: Base2.csv
Creado: Base3.csv
Creado: Base4.csv
Creado: Base5.csv
Creado: Base6.csv
Creado: Base7.csv
Creado: Base8.csv
Creado: Base9.csv
Creado: Base10.csv
Todos los archivos convertidos a CSV limpio


## Anexar los archivos separados (Base General)
>Se toma la decisión de Anexar por que las bases de datos comparten las misma columnas.



In [ ]:
# Ruta de tus CSV
path = "/content/drive/MyDrive/Proyecto Google/MaterialesProyectoFinal/MaterialesProyectoFinal/csv_limpios/*.csv"
files = glob.glob(path)

dfs = []
for file in files:
    df = pd.read_csv(file)
    dfs.append(df)

# Concatenamos todo
df_general = pd.concat(dfs, ignore_index=True)
print("Tamaño total de la base general:", df_general.shape)
df_general.head()

Tamaño total de la base general: (250000, 8)


,id_venta,fecha_hora,puesto,producto,categoria,canal,metodo_pago,precio
0,100001,07-02-2025 11:17 AM,Observatorio_1,Tamal verde,Tamal,Pedido anticipado,Tarjeta,25
1,100002,06/07/2025 10:27,SantaFe_2,Verde,Tamal,Pedido anticipado,Tarjeta,25
2,100003,"""2025-08-05 07:45:00""",Pantitlan_1,Verde,Tamal,Puesto,"""Efectivo""",25
3,100003,2025-08-05 07:45:00,Pantitlan_1,Bolillo,Complemento,Puesto,Efectivo,6
4,100003,2025-08-05 07:45:00,Pantitlan_1,Café,"""Bebida""",Puesto,Efectivo,12


## Información General.

In [ ]:
df_general.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   id_venta     250000 non-null  object
 1   fecha_hora   250000 non-null  object
 2   puesto       250000 non-null  object
 3   producto     250000 non-null  object
 4   categoria    250000 non-null  object
 5   canal        250000 non-null  object
 6   metodo_pago  250000 non-null  object
 7   precio       250000 non-null  object
dtypes: object(8)
memory usage: 15.3+ MB


# Analización de variables

In [ ]:
print("----Fecha_hora----")

puestos_total = df_general["fecha_hora"]

puestos_unicos = (
    puestos_total
    .dropna()
    .drop_duplicates()
    .sort_values()
)

display(puestos_unicos)

----Fecha_hora----


,fecha_hora
10446,"""01/07/2025 08:10"""
112620,"""01/07/2025 09:06"""
100490,"""01/07/2025 10:59"""
17911,"""01/07/2025 11:40"""
110563,"""01/08/2025 06:16"""
...,...
74419,31/12/2025 11:29
50303,31/12/2025 11:40
60357,31/12/2025 11:45
55150,31/12/2025 11:47


In [ ]:

print("----Puestos----")

puestos_total = df_general["puesto"]

puestos_unicos = (
    puestos_total
    .dropna()
    .drop_duplicates()
    .sort_values()
)

display(puestos_unicos)


----Puestos----


,puesto
390,"""CU_1"""
30,"""CU_2"""
195,"""CentroHistorico_1"""
13,"""Condesa_1"""
19,"""Coyoacan_1"""
204,"""DelValle_1"""
43,"""IndiosVerdes_1"""
566,"""InsurgentesSur_1"""
244,"""Iztapalapa_1"""
381,"""Lindavista_1"""


In [ ]:

print("----Producto----")
puestos_total = df_general["producto"]

puestos_unicos = (
    puestos_total
    .dropna()
    .drop_duplicates()
    .sort_values()
)

display(puestos_unicos)

----Producto----


,producto
263,"""Atole caliente"""
215,"""Atole"""
34,"""Bolillo"""
1131,"""Cafe"""
94,"""Café"""
212,"""Rojito"""
167,"""Tamal gourmet"""
20,"""Tamal mole"""
89,"""Tamal rojo"""
5,"""Tamal verde"""


In [ ]:
print("----categoria----")

puestos_total = df_general["categoria"]

puestos_unicos = (
    puestos_total
    .dropna()
    .drop_duplicates()
    .sort_values()
)
display(puestos_unicos)

----categoria----


,categoria
4,"""Bebida"""
9,"""Complemento"""
22,"""Tamal"""
12,Bebida
3,Complemento
0,Tamal


In [ ]:
print("----canal----")

puestos_total = df_general["canal"]

puestos_unicos = (
    puestos_total
    .dropna()
    .drop_duplicates()
    .sort_values()
)

display(puestos_unicos)

----canal----


,canal
15,"""Delivery"""
9,"""Pedido anticipado"""
77,"""Puesto"""
5,Delivery
0,Pedido anticipado
2,Puesto


In [ ]:
print("----metodo_pago----")

puestos_total = df_general["metodo_pago"]

puestos_unicos = (
    puestos_total
    .dropna()
    .drop_duplicates()
    .sort_values()
)
display(puestos_unicos)

----metodo_pago----


,metodo_pago
2,"""Efectivo"""
15,"""Tarjeta"""
54,"""Transferencia"""
3,Efectivo
0,Tarjeta
10,Transferencia


## Limpieza de Datos


In [ ]:
# Quitar comillas en las columnas

import pandas as pd

df_general = df_general.applymap(lambda x: x.replace('"', '') if isinstance(x, str) else x)

df_general.head(10)


/tmp/ipython-input-4065005016.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_general = df_general.applymap(lambda x: x.replace('"', '') if isinstance(x, str) else x)


,id_venta,fecha_hora,puesto,producto,categoria,canal,metodo_pago,precio
0,100001,07-02-2025 11:17 AM,Observatorio_1,Tamal verde,Tamal,Pedido anticipado,Tarjeta,25
1,100002,06/07/2025 10:27,SantaFe_2,Verde,Tamal,Pedido anticipado,Tarjeta,25
2,100003,2025-08-05 07:45:00,Pantitlan_1,Verde,Tamal,Puesto,Efectivo,25
3,100003,2025-08-05 07:45:00,Pantitlan_1,Bolillo,Complemento,Puesto,Efectivo,6
4,100003,2025-08-05 07:45:00,Pantitlan_1,Café,Bebida,Puesto,Efectivo,12
5,100004,07-22-2025 08:09 AM,RomaNorte_1,Tamal verde,Tamal,Delivery,Efectivo,25
6,100004,07-22-2025 08:09 AM,RomaNorte_1,Bolillo,Complemento,Delivery,Efectivo,6
7,100005,08-24-2025 08:38 AM,Lindavista_1,Tamal rojo,Tamal,Pedido anticipado,Efectivo,26
8,100006,2025-07-06 10:18:00,Tlalpan_1,Tamal gourmet,Tamal,Pedido anticipado,Efectivo,35
9,100006,2025-07-06 10:18:00,Tlalpan_1,Bolillo,Complemento,Pedido anticipado,Efectivo,6


In [ ]:
df_general["fecha_hora"].head()

,fecha_hora
0,07-02-2025 11:17 AM
1,06/07/2025 10:27
2,2025-08-05 07:45:00
3,2025-08-05 07:45:00
4,2025-08-05 07:45:00


In [ ]:
df_general.loc[df_general["fecha_hora"].isna(), "fecha_hora"].head(20)


,fecha_hora


In [ ]:
df_general["fecha_hora"].notna().sum()


np.int64(250000)

In [ ]:

#" Lipizar Fechas"

# =========================
# 0. Normalizar texto
# =========================
fecha_raw = (
    df_general["fecha_hora"]
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

# Inicializar columna destino
df_general["fecha_hora"] = pd.NaT


# =========================
# 1. ISO YYYY-MM-DD HH:MM:SS
# =========================
mask_iso = fecha_raw.str.match(
    r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$",
    na=False
)

df_general.loc[mask_iso, "fecha_hora"] = pd.to_datetime(
    fecha_raw[mask_iso],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)


# =========================
# 2. AM/PM MM-DD-YYYY
# =========================
mask_ampm = fecha_raw.str.contains(r'\bAM\b|\bPM\b', regex=True, na=False)

df_general.loc[mask_ampm, "fecha_hora"] = pd.to_datetime(
    fecha_raw[mask_ampm],
    format="%m-%d-%Y %I:%M %p",
    errors="coerce"
)


# =========================
# 3. AM/PM DD-MM-YYYY (fallback)
# =========================
mask_faltantes = mask_ampm & df_general["fecha_hora"].isna()

df_general.loc[mask_faltantes, "fecha_hora"] = pd.to_datetime(
    fecha_raw[mask_faltantes],
    format="%d-%m-%Y %I:%M %p",
    errors="coerce"
)


# =========================
# 4. Resto de formatos
# (DD/MM/YYYY, ISO corto, etc.)
# =========================
mask_rest = df_general["fecha_hora"].isna()

df_general.loc[mask_rest, "fecha_hora"] = pd.to_datetime(
    fecha_raw[mask_rest],
    dayfirst=True,
    errors="coerce"
)


# =========================
# 5. Diagnóstico final
# =========================
print("Tipo final:", df_general["fecha_hora"].dtype)
print("NaT restantes:", df_general["fecha_hora"].isna().sum())

df_general["fecha_hora"].head(10)





Tipo final: datetime64[ns]
NaT restantes: 0


,fecha_hora
0,2025-07-02 11:17:00
1,2025-07-06 10:27:00
2,2025-08-05 07:45:00
3,2025-08-05 07:45:00
4,2025-08-05 07:45:00
5,2025-07-22 08:09:00
6,2025-07-22 08:09:00
7,2025-08-24 08:38:00
8,2025-07-06 10:18:00
9,2025-07-06 10:18:00


In [ ]:
# "Limpiar columna Puesto"
df_general["puesto"]=(
    df_general["puesto"]
    .astype(str)
    .str.split("_")
    .str[0]
)

df_general["puesto"].unique()


array(['Observatorio', 'SantaFe', 'Pantitlan', 'RomaNorte', 'Lindavista',
       'Tlalpan', 'Tacubaya', 'Condesa', 'Coyoacan', 'Polanco', 'CU',
       'CentroHistorico', 'InsurgentesSur', 'IndiosVerdes', 'Reforma',
       'DelValle', 'Iztapalapa', 'Xochimilco'], dtype=object)

In [ ]:
# Puesto
homologación_puesto={

    'SantaFe':'Santa Fe' ,
    'RomaNorte':'Roma Norte',
    'CU':'Ciudad Universidad',
    'CentroHistorico':'Centro Histórico',
    'LaPaz':'La Paz',
    'InsurgentesSur':'Insurgentes Sur',
    'IndiosVerdes':'Indios Verdes',
    'DelValle':'Del Valle'

    }

df_general["puesto"]=(
    df_general["puesto"]
    .replace(homologación_puesto)
)

# Homologar  Producto

homologación_puesto={
    "Atole caliente":"Atole",
    "Cafe":"Café",
    "Rojito": "Tamal rojo",
    "Verdecito":"Tamal verde",
    "Rojo":"Tamal rojo",
    "Verde":"Tamal verde",
    "Bolillo":"Guajolota"
}


df_general["producto"]=(
    df_general["producto"]
    .replace(homologación_puesto)
)

df_general.head(10)

,id_venta,fecha_hora,puesto,producto,categoria,canal,metodo_pago,precio
0,100001,2025-07-02 11:17:00,Observatorio,Tamal verde,Tamal,Pedido anticipado,Tarjeta,25
1,100002,2025-07-06 10:27:00,Santa Fe,Tamal verde,Tamal,Pedido anticipado,Tarjeta,25
2,100003,2025-08-05 07:45:00,Pantitlan,Tamal verde,Tamal,Puesto,Efectivo,25
3,100003,2025-08-05 07:45:00,Pantitlan,Guajolota,Complemento,Puesto,Efectivo,6
4,100003,2025-08-05 07:45:00,Pantitlan,Café,Bebida,Puesto,Efectivo,12
5,100004,2025-07-22 08:09:00,Roma Norte,Tamal verde,Tamal,Delivery,Efectivo,25
6,100004,2025-07-22 08:09:00,Roma Norte,Guajolota,Complemento,Delivery,Efectivo,6
7,100005,2025-08-24 08:38:00,Lindavista,Tamal rojo,Tamal,Pedido anticipado,Efectivo,26
8,100006,2025-07-06 10:18:00,Tlalpan,Tamal gourmet,Tamal,Pedido anticipado,Efectivo,35
9,100006,2025-07-06 10:18:00,Tlalpan,Guajolota,Complemento,Pedido anticipado,Efectivo,6


In [ ]:
#"id_venta"
df_general["id_venta"]=pd.to_numeric(df_general["id_venta"],errors="coerce").astype("Int64")

#puesto
df_general["puesto"]=(df_general['puesto'].astype(str).str.strip())

#producto
df_general["producto"]=(df_general['producto'].astype(str).str.strip())

#categoria
df_general["categoria"]=(df_general['categoria'].astype(str).str.strip())

#canal
df_general["canal"]=(df_general['canal'].astype(str).str.strip())

#metodo_pago
df_general["metodo_pago"]=(df_general['metodo_pago'].astype(str).str.strip())

#Precio

df_general["precio"]=pd.to_numeric(df_general["precio"],errors="coerce").astype("float64")





In [ ]:
df_general.dtypes


,0
id_venta,Int64
fecha_hora,datetime64[ns]
puesto,object
producto,object
categoria,object
canal,object
metodo_pago,object
precio,float64


In [ ]:

ruta_salida ="/content/drive/MyDrive/Proyecto Google/MaterialesProyectoFinal/MaterialesProyectoFinal/csv_limpios/Base general.csv"
df_general.to_csv(
    ruta_salida,
    index=False,

)


## Cargar al Bucket

In [ ]:
cliente=storage.Client(project=project_ID)
bucket=cliente.bucket(BUCKET_NAME)
blob= bucket.blob(Destino)
blob.upload_from_filename(ruta_salida)

print("df_general cargado al bucket correctamente")

df_general cargado al bucket correctamente
